# 2 - IP Prune NetMap Reaches to 5km and Clip by HUC8 Extents


The intended use of this tool is to take a high resolution NetMap dataset (which contains multiple stream reach line features between confluences) and reduce the resolution of the dataset to contain only one stream reach line feature between confluences. These pre-processed reaches will then be clipped to the extent of National Hydrography Dataset (NHD) HUC 8 polygons. (NHD data can be downloaded here: https://www.usgs.gov/national-hydrography/access-national-hydrography-products)

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.


## Required Inputs:

- A geodatabase containing both a) the unprocessed, high resolution NetMap synthetic stream reach dataset, which must have "reach" prefix in the feature class name, and b) NHD HUC8 polygons which overlay the synthetic stream reaches, which must have "HUC8" suffix in the feature class name.


## Geoprocessing Output:

- A pre-processed stream reach feature class, saved in the input geodatbase with a "_5km_" suffix. (The suffix will allow further processing of the feature classes using only the geodatabase as input in other tools.)

- A collection of pre-processed stream reach feature classes clipped by HUC 8 extent, saved in the input geodatbase with a suffix indicating the unique HUC 8 ID number, followed by "HUC8". (The suffix will allow further processing of the feature classes using only the geodatabase as input in other tools.)

## Processing Steps:

1. Load the large NetMap dataset and the HUC8 polygons as feature layers, searching the input geodatabase for feature classes starting with "reach" or ending with "HUC8".
2. Find the field beginning with "AREA", and use that field to select all reaches with >5km upstream area. Copy them to a new feature layer.
3. Dissolve all lines in the new feature layer into a single polyline feature.
4. Create points at all intersections.
5. Split the polyline at the points.
6. Flip the direction of streamlines (if necessary).
7. For each HUC 8 polygon in the GDB feature class, clip the stream network and save as an individual polyline feature classes in the geodatabase.
8. Add a field to the new feature class, and name each new confluence to confluence reach with a unique ID equivalent to the OID value. (These will become RCA IDS.)
9. Delete all intermediary files and layers.

### Code starts here:

#### Setup

Import modules and set environment to the user-provided geodatabase and workspace/scratch directory. Additionally, allow  the addition of intermediary outputs to the ArcPro project map.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [1]:
import arcpy
from datetime import datetime

work = "F:/GIS/IP/Kuskokwim.gdb"

#work = arcpy.GetParameterAsText(0)

arcpy.env.addOutputsToMap = True

arcpy.env.workspace = work

List feature classes containing the required prefix (for reaches) and suffix (for HUC8 polygons).  Create a date tag to be used in naming temporary files. This will help prevent locking errors when trying to use temporary files in loops or when deleting temporary files after loops have run.

In [2]:
featureclasses = arcpy.ListFeatureClasses()

for fc in featureclasses:
    if fc.startswith("reach") == True:
        reach = fc
    elif fc.endswith("HUC8") == True:
        huc8 = fc
    else:
        pass

    
dtag = datetime.now().strftime("%Y%m%d%H%M%S")
reach_ = "reach_" + dtag

dtag = datetime.now().strftime("%Y%m%d%H%M%S")
huc8_ = "huc8_" + dtag  

Create feature layers from the reaches and HUC8 polygons.

In [3]:
arcpy.management.MakeFeatureLayer(reach, reach_)
arcpy.management.MakeFeatureLayer(huc8, huc8_)

<Result 'huc8_20220802172407'>

Find the area field in the reach feature layer, then use it to select reaches with greater than 5km catchment area.

In [4]:
fields = arcpy.ListFields(reach_)

for field in fields:
    if field.name.startswith("AREA") == True:
        f = field.name
    else:
        pass
f

'AREA_SQKM'

In [5]:
f_Delimited = arcpy.AddFieldDelimiters(reach_, f)
where = str(f_Delimited + ' > 5')
arcpy.management.SelectLayerByAttribute(reach_, "NEW_SELECTION", where, "")

id,value
0,a Layer object
1,531473


Copy the selected reaches to a new feature class.

In [6]:
out5km = str(reach + "_5km")
arcpy.management.CopyFeatures(reach_, out5km)

<Result 'F:/GIS/IP/Kuskokwim.gdb\\reach_Kuskokwim_5km'>

Dissolve all streamlines to one feature.

In [7]:
out5km_d = str(out5km + "_dissolved")
arcpy.Dissolve_management(out5km, out5km_d, "", "", "MULTI_PART", "DISSOLVE_LINES")

<Result 'F:/GIS/IP/Kuskokwim.gdb\\reach_Kuskokwim_5km_dissolved'>

Create a point at each vertex (ie, each confluence in the stream network).

In [8]:
out5km_d_p = str(out5km_d + "_pts")
arcpy.FeatureVerticesToPoints_management(out5km_d, out5km_d_p, "BOTH_ENDS")

<Result 'F:/GIS/IP/Kuskokwim.gdb\\reach_Kuskokwim_5km_dissolved_pts'>

Split the dissolved streamline at each confluence point.

In [9]:
out5km_d_p_s = str(out5km_d_p + "_split")
arcpy.SplitLineAtPoint_management(out5km_d, out5km_d_p, out5km_d_p_s, "")

<Result 'F:/GIS/IP/Kuskokwim.gdb\\reach_Kuskokwim_5km_dissolved_pts_split'>

Copy the output, and flip the direction of the streamlines (by default, NetMap reaches are oriented in the opposite direction when read in ArcPro.)

In [10]:
out5km_ = str(out5km + "_")

arcpy.CopyFeatures_management(out5km_d_p_s, out5km_)

arcpy.FlipLine_edit(out5km_)

<Result 'reach_Kuskokwim_5km_'>

Prevent the following code block from adding outputs to the map. Define the Object ID field in the HUC8 feature layer, and use it to loop through each polygon feature in the layer. 

For each feature, create a temporary polygon output (with a date tag in its name) and select all confluence-to-confluence reaches than have their centers in that polygon. Copy the selected features to a new feature class named using the HUC8 ID and an additional "HUC8" suffix. Add a field for RCA ID and calculate it as equivalent to the Object ID. Finally, delete the temporary HUC8 polygon before moving on to the next feature.

In this way, the user-provided geodatabase will end up with an individual, non-overlapping, pre-processed stream network for each HUC 8 polygon. The "_ HUC8" suffixes will allow further processing of all stream networks within the geodatabase without providing specific feature class names.

In [11]:
arcpy.env.addOutputsToMap = False

oid_field = arcpy.Describe(huc8_).OIDFieldName
oid_Delimited = arcpy.AddFieldDelimiters(huc8_, oid_field)

huc8_field = "huc8"

with arcpy.da.SearchCursor(huc8_, [oid_field, huc8_field]) as cursor:
    for row in cursor:
        
        o = row[0]
        where = str(oid_Delimited + ' = ' + str(o)) 
        
        dtag = datetime.now().strftime("%Y%m%d%H%M%S")
        huc8_temp = "HUC8_TEMP_" + dtag
    
        arcpy.management.MakeFeatureLayer(huc8, huc8_temp, where)
        
        arcpy.management.SelectLayerByLocation(out5km_, "HAVE_THEIR_CENTER_IN", huc8_temp, "", "NEW_SELECTION", "")
        
        name = str(out5km + "_" + str(row[1]) + "_HUC8")
        arcpy.CopyFeatures_management(out5km_, name)
        
        arcpy.management.AddField(name, "RCA_ID", "LONG", "", "", "", "")
        arcpy.management.CalculateField(name, "RCA_ID", "!OBJECTID!")
        
        arcpy.management.Delete(huc8_temp)

Delete each intermediary file, trying both its variable name and its full file path. This should clear all locked files in the geodatabase and the system folder workspace, but it is recommended to check these locations after running the script to be sure that all intermediary files have been deleted.

In [12]:
try:
    arcpy.management.Delete(reach_)
except:
    desc = arcpy.Describe(reach_)
    arcpy.management.Delete(desc.path)
    
try:
    arcpy.management.Delete(huc8_)
except:
    desc = arcpy.Describe(huc8_)
    arcpy.management.Delete(desc.path)

try:
    arcpy.management.Delete(out5km)
except:
    desc = arcpy.Describe(out5km)
    arcpy.management.Delete(desc.path)

try:
    arcpy.management.Delete(out5km_d)
except:
    desc = arcpy.Describe(out5km_d)
    arcpy.management.Delete(desc.path)
    
try:
    arcpy.management.Delete(out5km_d_p)
except:
    desc = arcpy.Describe(out5km_d_p)
    arcpy.management.Delete(desc.path)

try:
    arcpy.management.Delete(out5km_d_p_s)
except:
    desc = arcpy.Describe(out5km_d_p_s)
    arcpy.management.Delete(desc.path)
